In [ ]:
# Celula unica - Importacao de dados GPKG para PostgreSQL
import geopandas as gpd
import geoalchemy2
import geopandas as gpd
from sqlalchemy import create_engine
from sqlalchemy import text
from sqlalchemy import create_engine, text
from pathlib import Path

# =============
# CONFIGURACOES 
# =============

# Pasta onde estao os arquivos GPKG
PASTA_DADOS = Path(r"C:\Temp")  #ALTERAR PARA SUA PASTA. EX: "C:\Downloads"

# Nomes dos arquivos
CAR_ARQUIVO = "es_mt_car_20260406.gpkg"
SIGEF_ARQUIVO = "pa_br_sigef_privado_20260107.gpkg"

# Nomes das tabelas no banco
CAR_TABELA = "es_mt_car_20260406"
SIGEF_TABELA = "pa_br_sigef_privado_20260107"

# Credenciais do banco local
DB_HOST = "localhost"
DB_PORT = "5432"
DB_USER = "postgres"
DB_PASSWORD = "postgres"
DB_NAME = "geotec"

# ==========================================
# FUNCOES
# ==========================================

def conectar_banco():
    """Cria conexao com o banco"""
    engine = create_engine(f'postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}')
    return engine

def criar_schemas(engine):
    """Cria schemas car e incra se nao existirem"""
    with engine.connect() as conn:
        conn.execute(text("CREATE SCHEMA IF NOT EXISTS car"))
        conn.execute(text("CREATE SCHEMA IF NOT EXISTS incra"))
        conn.execute(text("CREATE EXTENSION IF NOT EXISTS postgis"))
        conn.commit()
    print("[OK] Schemas car e incra verificados/criados")

def verificar_tabela(engine, schema, tabela):
    """Verifica se a tabela ja existe no banco"""
    try:
        with engine.connect() as conn:
            result = conn.execute(text(f"SELECT 1 FROM {schema}.{tabela} LIMIT 1"))
            return True
    except:
        return False

def contar_registros(engine, schema, tabela):
    """Conta registros de uma tabela"""
    try:
        with engine.connect() as conn:
            result = conn.execute(text(f"SELECT COUNT(*) FROM {schema}.{tabela}"))
            return result.fetchone()[0]
    except:
        return 0

def importar_gpkg(engine, caminho_arquivo, schema, tabela):
    """Importa arquivo GPKG para o banco"""
    print(f"  Lendo arquivo: {caminho_arquivo.name}")
    gdf = gpd.read_file(str(caminho_arquivo))
    print(f"  Registros encontrados: {len(gdf)}")
    
    gdf.to_postgis(tabela, engine, schema=schema, if_exists="replace")
    print(f"  Importado para {schema}.{tabela}")
    
    return len(gdf)

# ==========================================
# MAIN
# ==========================================

def main():
    print("=" * 50)
    print("IMPORTACAO DE DADOS GEOPROCESSAMENTO")
    print("=" * 50)
    
    # Verificar se a pasta existe
    if not PASTA_DADOS.exists():
        print(f"\n[ERRO] Pasta nao encontrada: {PASTA_DADOS}")
        print("[INFO] Crie a pasta ou altere a variavel PASTA_DADOS")
        return
    
    # Verificar se os arquivos existem
    car_path = PASTA_DADOS / CAR_ARQUIVO
    sigef_path = PASTA_DADOS / SIGEF_ARQUIVO
    
    arquivos_faltando = []
    if not car_path.exists():
        arquivos_faltando.append(CAR_ARQUIVO)
    if not sigef_path.exists():
        arquivos_faltando.append(SIGEF_ARQUIVO)
    
    if arquivos_faltando:
        print(f"\n[ERRO] Arquivos nao encontrados: {arquivos_faltando}")
        print(f"[INFO] Coloque os arquivos em: {PASTA_DADOS}")
        return
    
    # Conectar ao banco
    print("\n[1] Conectando ao PostgreSQL...")
    try:
        engine = conectar_banco()
        print("[OK] Conexao estabelecida")
    except Exception as e:
        print(f"[ERRO] Falha na conexao: {e}")
        print("[INFO] Verifique se o PostgreSQL esta rodando")
        return
    
    # Criar schemas
    print("\n[2] Configurando schemas...")
    try:
        criar_schemas(engine)
    except Exception as e:
        print(f"[ERRO] Falha ao criar schemas: {e}")
        return
    
    # Importar CAR
    print("\n[3] Importando CAR...")
    if verificar_tabela(engine, "car", CAR_TABELA):
        registros = contar_registros(engine, "car", CAR_TABELA)
        print(f"[INFO] Tabela car.{CAR_TABELA} ja existe com {registros} registros")
        resposta = input("[?] Deseja recriar? (s/N): ")
        if resposta.lower() == 's':
            importar_gpkg(engine, car_path, "car", CAR_TABELA)
        else:
            print("  Mantendo dados existentes")
    else:
        importar_gpkg(engine, car_path, "car", CAR_TABELA)
    
    # Importar SIGEF
    print("\n[4] Importando SIGEF...")
    if verificar_tabela(engine, "incra", SIGEF_TABELA):
        registros = contar_registros(engine, "incra", SIGEF_TABELA)
        print(f"[INFO] Tabela incra.{SIGEF_TABELA} ja existe com {registros} registros")
        resposta = input("[?] Deseja recriar? (s/N): ")
        if resposta.lower() == 's':
            importar_gpkg(engine, sigef_path, "incra", SIGEF_TABELA)
        else:
            print("  Mantendo dados existentes")
    else:
        importar_gpkg(engine, sigef_path, "incra", SIGEF_TABELA)
    
    # Verificar resultados finais
    print("\n[5] Verificando dados importados...")
    car_count = contar_registros(engine, "car", CAR_TABELA)
    sigef_count = contar_registros(engine, "incra", SIGEF_TABELA)
    
    print(f"  CAR: {car_count} registros")
    print(f"  SIGEF: {sigef_count} registros")
    
    # Fechar conexao
    engine.dispose()
    
    print("\n" + "=" * 50)
    print("PRONTO! AGORA PODEMOS SEGUIR COM O CURSO")
    print("=" * 50)

# Executar
if __name__ == "__main__":
    main()